# D&D Beyond Feat Sync

Sync homebrew feats from Google Sheets to D&D Beyond.

## Setup Instructions

1. Get your D&D Beyond cookies (Copy as cURL and extract Cookie header).
2. Get security tokens from the create feat page.
3. Update your `.env` with `DDB_COOKIES`, `DDB_SECURITY_TOKEN`, and `DDB_AUTHENTICITY_TOKEN`.


## 1. Setup and Imports

In [ ]:
import pandas as pd
import requests
import json
import time
import re
import os
from typing import Dict, List, Optional
from dotenv import load_dotenv
from datetime import datetime
from pathlib import Path
import sys

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from FiveETools.gsheets_client import ContentSheetsClient

from DNDBeyond.helpers.DnDBeyondAPI import DnDBeyondAPI
from DNDBeyond.helpers import convert_feat_to_ddb, normalize_ddb_id

print("✓ Imports loaded")


## 2. D&D Beyond Authentication

In [ ]:
# ============================================
# D&D Beyond Configuration
# ============================================

load_dotenv()

DDB_BASE_URL = os.getenv("DDB_BASE_URL") or "https://www.dndbeyond.com"
DDB_COOKIES = os.getenv("DDB_COOKIES")
DDB_SECURITY_TOKEN = os.getenv("DDB_SECURITY_TOKEN")
DDB_AUTHENTICITY_TOKEN = os.getenv("DDB_AUTHENTICITY_TOKEN")
REQUEST_VERIFICATION_TOKEN = os.getenv("REQUEST_VERIFICATION_TOKEN")
DDB_USER_ID = os.getenv("DDB_USER_ID")
DDB_USERNAME = os.getenv("DDB_USERNAME")

# ============================================
# Setup Session
# ============================================

session = requests.Session()

if DDB_COOKIES:
    session.headers.update({
        "Cookie": DDB_COOKIES,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:147.0) Gecko/20100101 Firefox/147.0",
        "Referer": f"{DDB_BASE_URL}/homebrew/creations/create-feat/create",
        "Origin": DDB_BASE_URL,
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1"
    })

    print("✓ Using cookie authentication")
    print(f"✓ Base URL: {DDB_BASE_URL}")
    print(f"✓ User: {DDB_USERNAME} (ID: {DDB_USER_ID})")

    if DDB_SECURITY_TOKEN and DDB_AUTHENTICITY_TOKEN:
        print("✓ Security tokens configured")
    else:
        print("⚠️  Security tokens not set - you'll need these to create feats!")
else:
    print("✗ ERROR: No cookies provided!")
    print("   Please set DDB_COOKIES with your browser session cookies")


## 3. Load Feat Data

In [ ]:
print("Loading feats from Google Sheets...")
client = ContentSheetsClient("non-fantasy", credentials_path=str(repo_root / "FiveETools" / "key.json"))
df_feats = client.get_sheet_by_name("feats")

feat_list = []
for _, row in df_feats.iterrows():
    name = row.get("Name")
    source = row.get("Source")
    if not name or pd.isna(name):
        continue
    if pd.notna(source) and str(source).strip() == '':
        continue
    # Build entries list similar to 5etools format
    entries = []
    flavor = row.get("Flavor Text")
    if pd.notna(flavor) and str(flavor).strip():
        entries.append(str(flavor))
    # Include remaining columns from Feat onwards
    if 'Feat' in df_feats.columns:
        feat_pos = df_feats.columns.get_loc('Feat')
        entries.extend([str(v) for v in row.iloc[feat_pos:].dropna().tolist()])
    feat_list.append({
        'name': str(name),
        'entries': entries,
    })

print(f"✓ Loaded {len(feat_list)} feats")
if feat_list:
    print(f"Sample feat: {feat_list[0].get('name')}")


## 4. D&D Beyond API Helper

In [ ]:
ddb_api = DnDBeyondAPI(
    session,
    DDB_SECURITY_TOKEN,
    DDB_AUTHENTICITY_TOKEN,
    verification_token=REQUEST_VERIFICATION_TOKEN,
)
print("✓ D&D Beyond API initialized")


## 5. Sync Feats to D&D Beyond

In [ ]:
DRY_RUN = False
BATCH_SIZE = 1
DELAY = 1
VERBOSE = True
SKIP_EXISTING = True
UPDATE_EXISTING = True

print("=" * 60)
if DRY_RUN:
    print("DRY RUN - No feats will be created/updated")
else:
    print("⚠️  LIVE MODE - Feats WILL be created/updated!")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Batch size: {'ALL FEATS' if BATCH_SIZE is None else BATCH_SIZE}")
print(f"Delay between requests: {DELAY}s")
print(f"Verbose logging: {'ON' if VERBOSE else 'OFF'}")
print(f"Skip existing: {'YES' if SKIP_EXISTING else 'NO (will attempt to create anyway)'}")
print(f"Update existing: {'YES (will update when DDB ID exists)' if UPDATE_EXISTING else 'NO'}")
print()

feats_to_sync = feat_list if BATCH_SIZE is None else feat_list[:BATCH_SIZE]
print(f"Total feats to process: {len(feats_to_sync)}")
print()

print("Checking spreadsheet for existing DDB IDs...")
FEAT_NAME_COLUMN = "Name"
feat_to_ddb_id = {}
if 'DDB' in df_feats.columns and FEAT_NAME_COLUMN in df_feats.columns:
    for _, row in df_feats.iterrows():
        feat_name = row.get(FEAT_NAME_COLUMN)
        ddb_id = row.get('DDB')
        if feat_name and pd.notna(ddb_id):
            normalized_id = normalize_ddb_id(ddb_id)
            if normalized_id:
                feat_to_ddb_id[feat_name] = normalized_id
    print(f"✓ Found {len(feat_to_ddb_id)} feats with DDB IDs in spreadsheet")
else:
    print("  No DDB IDs found in spreadsheet")

print("\nFetching existing homebrew feats from D&D Beyond...")
if not DRY_RUN:
    try:
        existing_feats = ddb_api.feats.list()
        print(f"✓ Found {len(existing_feats)} existing homebrew feats\n")
    except Exception as exc:
        print(f"⚠️  Could not fetch existing feats: {exc}")
        existing_feats = []
else:
    existing_feats = []

REQUIRED_FORM_FIELDS = set()
if not DRY_RUN:
    try:
        response = session.get(f"{DDB_BASE_URL}/homebrew/creations/create-feat/create", timeout=30)
        REQUIRED_FORM_FIELDS = set(re.findall(r'name="([^"]+)"[^>]*data-validation-required="true"', response.text))
        print("✓ Refreshed required feat fields from live page")
    except Exception as exc:
        print(f"⚠️  Could not refresh required feat fields: {exc}")

results = {
    'created': 0,
    'updated': 0,
    'skipped': 0,
    'errors': 0,
    'details': []
}

spreadsheet_updates = []

for i, feat in enumerate(feats_to_sync, 1):
    feat_name = feat.get('name', 'Unnamed')
    print(f"[{i}/{len(feats_to_sync)}] Processing: {feat_name}")

    try:
        if VERBOSE:
            print("  → Converting feat to D&D Beyond format...")
        try:
            ddb_form = convert_feat_to_ddb(feat)
            if VERBOSE:
                print(f"    Description length: {len(ddb_form.get('item-description-wysiwyg', ''))}")
        except Exception as conv_error:
            print(f"  ✗ Conversion failed: {conv_error}")
            results['errors'] += 1
            results['details'].append({
                'title': feat_name,
                'action': 'error',
                'success': False,
                'error': f"Conversion error: {str(conv_error)}",
                'error_type': 'conversion'
            })
            continue

        missing_fields = [
            key for key in REQUIRED_FORM_FIELDS
            if key not in ddb_form or str(ddb_form.get(key, '')) == ''
        ]
        if missing_fields:
            print("  ✗ Missing required fields for create")
            print(f"    Missing fields: {missing_fields}")
            results['errors'] += 1
            results['details'].append({
                'title': feat_name,
                'action': 'error',
                'success': False,
                'error_type': 'validation_failed',
                'missing_fields': missing_fields
            })
            continue

        existing_id = feat_to_ddb_id.get(feat_name)
        if existing_id:
            if UPDATE_EXISTING and not SKIP_EXISTING:
                if DRY_RUN:
                    print(f"  → DRY RUN: Would update feat (ID: {existing_id})")
                    results['updated'] += 1
                    results['details'].append({
                        'title': feat_name,
                        'action': 'dry_run_update',
                        'success': True,
                        'id': existing_id
                    })
                else:
                    print(f"  → Updating existing feat (ID: {existing_id})")
                    slug = DnDBeyondAPI.create_slug(feat_name)
                    updated = ddb_api.feats.update(existing_id, slug, {'form_data': ddb_form})
                    if updated:
                        print("  ✓ Updated feat successfully")
                        results['updated'] += 1
                        results['details'].append({
                            'title': feat_name,
                            'action': 'updated',
                            'success': True,
                            'id': existing_id
                        })
                    else:
                        print("  ✗ Failed to update feat")
                        results['errors'] += 1
                        results['details'].append({
                            'title': feat_name,
                            'action': 'error',
                            'success': False,
                            'error_type': 'update_failed',
                            'id': existing_id,
                            'error': ddb_api.last_error
                        })
                if i < len(feats_to_sync):
                    time.sleep(DELAY)
            else:
                print(f"  ✓ Already synced (DDB ID: {existing_id})")
                results['skipped'] += 1
                results['details'].append({
                    'title': feat_name,
                    'action': 'skipped',
                    'success': True,
                    'id': existing_id,
                    'reason': 'already_in_spreadsheet'
                })
            continue

        if SKIP_EXISTING and not DRY_RUN and existing_feats:
            existing_id = ddb_api.feats.find_by_name(feat_name, existing_feats)
            if existing_id:
                print(f"  ⚠️  Feat exists on D&D Beyond (ID: {existing_id})")
                spreadsheet_updates.append({
                    'match_value': feat_name,
                    'update_column': 'DDB',
                    'update_value': existing_id
                })
                results['skipped'] += 1
                results['details'].append({
                    'title': feat_name,
                    'action': 'skipped',
                    'success': True,
                    'id': existing_id,
                    'reason': 'already_exists_ddb'
                })
                continue

        if DRY_RUN:
            print("  → DRY RUN: Would create feat")
            results['created'] += 1
            results['details'].append({
                'title': feat_name,
                'action': 'dry_run_create',
                'success': True
            })
        else:
            print("  → Creating feat on D&D Beyond...")
            feat_id = ddb_api.feats.create({'form_data': ddb_form})
            if feat_id:
                print(f"  ✓ Created: ID {feat_id}")
                spreadsheet_updates.append({
                    'match_value': feat_name,
                    'update_column': 'DDB',
                    'update_value': feat_id
                })
                results['created'] += 1
                results['details'].append({
                    'title': feat_name,
                    'action': 'created',
                    'success': True,
                    'id': feat_id
                })
                existing_feats.append({'name': feat_name, 'id': feat_id})
            else:
                print("  ✗ Failed to create feat")
                results['errors'] += 1
                results['details'].append({
                    'title': feat_name,
                    'action': 'error',
                    'success': False,
                    'error_type': 'creation_failed',
                    'error': ddb_api.last_error
                })
            if i < len(feats_to_sync):
                time.sleep(DELAY)

    except Exception as exc:
        print(f"  ✗ Unexpected error: {exc}")
        results['errors'] += 1
        results['details'].append({
            'title': feat_name,
            'action': 'error',
            'success': False,
            'error_type': 'unexpected_exception',
            'error': str(exc)
        })

# Write DDB IDs back to spreadsheet
if spreadsheet_updates and not DRY_RUN:
    print('\n' + '=' * 60)
    print('Updating Google Sheets with DDB IDs...')
    print('=' * 60)
    try:
        update_results = client.batch_update_cells_by_row_match(
            ContentSheetsClient.SHEET_GIDS['non-fantasy']['feats'],
            FEAT_NAME_COLUMN,
            spreadsheet_updates
        )
        success_count = sum(1 for v in update_results.values() if v)
        print(f"✓ Updated {success_count}/{len(spreadsheet_updates)} feats in spreadsheet")
    except Exception as exc:
        print(f"✗ Error updating spreadsheet: {exc}")

print()
print('=' * 60)
print('SYNC COMPLETE')
print('=' * 60)
if DRY_RUN:
    print(f"DRY RUN: Would create {results['created']} feats")
    print(f"DRY RUN: Would update {results['updated']} feats")
else:
    print(f"✓ Created: {results['created']}")
    print(f"✓ Updated: {results['updated']}")
    print(f"⚠️  Skipped (already exist): {results['skipped']}")
    print(f"✗ Errors: {results['errors']}")

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_file = f"out/feat_sync_log_{timestamp}.json"
log_dir = os.path.dirname(log_file)
os.makedirs(log_dir, exist_ok=True)
with open(log_file, 'w') as f:
    json.dump({
        'timestamp': timestamp,
        'dry_run': DRY_RUN,
        'verbose': VERBOSE,
        'batch_size': BATCH_SIZE,
        'skip_existing': SKIP_EXISTING,
        'update_existing': UPDATE_EXISTING,
        'total': len(feats_to_sync),
        'created': results['created'],
        'updated': results['updated'],
        'skipped': results['skipped'],
        'errors': results['errors'],
        'details': results['details']
    }, f, indent=2)

print(f"\n✓ Log saved to: {log_file}")
